In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

In [ ]:
# Install requirements
#%pip install -r ../requirements.txt
%pip install -q bitsandbytes accelerate
%pip install groq

In [ ]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q "faiss-cpu"

# 2. Query Rewriting
Triển khai Multi-query, HyDE, và xử lý code-switching

In [ ]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from src.rewriting.query_rewriter import QueryRewriter
from src.generation.llm_generator import LLMGenerator
from src.data.data_loader import DataLoader

In [ ]:
# Initialize
llm = LLMGenerator(provider="groq", groq_api_key=GROQ_API_KEY, groq_model="llama-3.1-8b-instant")
data_loader = DataLoader("../data")

In [ ]:
rewriter = QueryRewriter(llm)

## Test Multi-Query Rewriting

In [ ]:
queries = [
    "Bệnh đái tháo đường là gì?",
    "Machine learning là gì?",
    "Cách phòng chống cúm"
]

import time

for query in queries:
    rewritten = rewriter.rewrite(query, method="multi-query-hyde")
    time.sleep(1.2)
    print(f"Original: {query}")
    print(f"Rewritten ({len(rewritten)} queries):")
    for i, q in enumerate(rewritten, 1):
        print(f"  {i}. {q}")
    print()

## Test Code-Switching Detection

In [ ]:
code_switch_queries = [
    "Machine learning là gì?",
    "Cách setup Docker trên Ubuntu",
    "API là gì trong lập trình",
    "Triệu chứng bệnh cúm"
]

for query in code_switch_queries:
    has_switch = rewriter.has_code_switching(query)
    print(f"{query}")
    print(f"  Code-switching detected: {has_switch}")
    if has_switch:
        variations = rewriter.code_switching_handle(query)
        print(f"  Variations: {variations}")
    print()

## Evaluate Query Rewriting on Dataset

In [ ]:
from tqdm.auto import tqdm
import time

# Danh sách các dataset cần đánh giá
datasets_to_eval = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]
num_samples_per_ds = 50

summary_results = {}

print(f"Bắt đầu đánh giá Query Rewriting trên {len(datasets_to_eval)} datasets...")

# Thanh tiến trình tổng cho các Datasets
dataset_pbar = tqdm(datasets_to_eval, desc="Tổng quát", unit="dataset")

for ds_name in dataset_pbar:
    try:
        # Cập nhật trạng thái thanh tiến trình tổng
        dataset_pbar.set_description(f"Đang xử lý: {ds_name}")

        data = data_loader.load_dataset(ds_name, "test")[:num_samples_per_ds]
        if not data:
            continue

        rewritten_counts = []
        cs_detected_count = 0

        # Thanh tiến trình chi tiết cho từng mẫu trong dataset
        sample_pbar = tqdm(data, desc=f"   └─ {ds_name}", leave=False, unit="sample")

        for sample in sample_pbar:
            query = sample.get("question") or sample.get("query") or ""
            if not query:
                continue

            # Kiểm tra code-switching
            if rewriter.has_code_switching(query):
                cs_detected_count += 1

            # Gọi pipeline rewrite (LLM sẽ chạy ở đây)
            rewritten = rewriter.rewrite(query, method="multi-query-hyde")
            rewritten_counts.append(len(rewritten))

            # Cập nhật thống kê nhanh lên thanh tiến trình
            sample_pbar.set_postfix({"CS_Found": cs_detected_count, "Variants": len(rewritten)})

        # Tính toán kết quả sau khi xong 1 dataset
        avg_rewritten = sum(rewritten_counts) / len(rewritten_counts) if rewritten_counts else 0
        cs_rate = (cs_detected_count / len(data)) * 100

        summary_results[ds_name] = {
            "avg_variants": avg_rewritten,
            "cs_detected_rate": cs_rate
        }

    except Exception as e:
        print(f"\nLỗi khi xử lý {ds_name}: {e}")

# In bảng tổng hợp cuối cùng
print("\n" + "="*60)
print(f"{'Dataset':<20} | {'Avg Variants':<15} | {'CS Rate (%)':<12}")
print("-" * 60)
for ds, metrics in summary_results.items():
    print(f"{ds:<20} | {metrics['avg_variants']:<15.2f} | {metrics['cs_detected_rate']:<12.1f}")
print("="*60)